In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_csv("Summary_of_Results_3_18_2026.csv", parse_dates=["FAILDATE"])
fed = pd.read_csv("avg Federal Funds Effective Rate.csv", parse_dates=["DATE"])
unemp = pd.read_csv("US_unemployment_rate.csv")  # YEAR, UNRATE



In [2]:
# Ensure date components exist
df["YEAR"] = df["FAILDATE"].dt.year
df["MONTH"] = df["FAILDATE"].dt.month

In [ ]:
# 2. LAGGED MACRO FEATURES

# --- A. FED RATE (Monthly) → M-1 lag ---
fed["YEAR"] = fed["DATE"].dt.year
fed["MONTH"] = fed["DATE"].dt.month

# Shift FED rate by 1 month
fed["FEDFUNDS_LAG1"] = fed["VALUE"].shift(1)

# Merge on YEAR + MONTH
df = df.merge(
    fed[["YEAR", "MONTH", "FEDFUNDS_LAG1"]],
    on=["YEAR", "MONTH"],
    how="left"
)

# --- B. UNEMPLOYMENT (Annual) → N-1 lag ---
unemp["UNRATE_LAG1"] = unemp["UNRATE"].shift(1)

df = df.merge(
    unemp[["YEAR", "UNRATE_LAG1"]],
    on="YEAR",
    how="left"
)

# -----------------------------
# 3. BANK-LEVEL ENGINEERED FEATURES
# -----------------------------

# Deposit-to-Asset Ratio
df["Deposit_to_Asset_Ratio"] = df["QBFDEP"] / df["QBFASSET"]

# Log-transformed size variables
df["log_assets"] = np.log1p(df["QBFASSET"])
df["log_deposits"] = np.log1p(df["QBFDEP"])

# LGD target
df["LGD"] = df["COST"] / df["QBFASSET"]

# -----------------------------
# 4. CRISIS INDICATORS
# -----------------------------
df["CRISIS"] = (
    df["YEAR"].between(1985, 1994) |
    df["YEAR"].between(2008, 2012)
).astype(int)

# -----------------------------
# 5. STATE-LEVEL SYSTEMIC STRESS
# -----------------------------

df = df.sort_values("FAILDATE")

df["State_Failures_Last_12M"] = (
    df.groupby("STATE")
      .rolling("365D", on="FAILDATE")
      .size()
      .reset_index(level=0, drop=True) - 1
)

df["State_Failures_Last_12M"] = df["State_Failures_Last_12M"].clip(lower=0)


# -----------------------------
# 6. CATEGORICAL ENCODING
# -----------------------------

categorical_cols = ['CHCLASS1', 'RESTYPE1', 'SAVR', 'STATE']
# 1. Macro variables
df['FEDFUNDS_LAG1'] = df['FEDFUNDS_LAG1'].fillna(method='bfill')
df['UNRATE_LAG1'] = df['UNRATE_LAG1'].fillna(method='bfill')

# 2. State-level stress
df['State_Failures_Last_12M'] = df['State_Failures_Last_12M'].fillna(0)

# 3. Categorical variables
df[categorical_cols] = df[categorical_cols].fillna("Unknown")

# 4. Drop rows where LGD cannot be computed
df = df.dropna(subset=['QBFASSET'])


df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# -----------------------------
# 7. FINAL TRAINING SET
# -----------------------------

feature_cols = [
    "Deposit_to_Asset_Ratio",
    "log_assets",
    "log_deposits",
    "FEDFUNDS_LAG1",
    "UNRATE_LAG1",
    "CRISIS",
    "State_Failures_Last_12M"
] + [col for col in df.columns if col.startswith(("RESTYPE1_", "CHCLASS1_", "STATE_"))]

X = df[feature_cols]
y = df["LGD"]

# -----------------------------
# 8. TIME-AWARE TRAIN/TEST SPLIT
# -----------------------------

# Sort by date to avoid leakage
df = df.sort_values("FAILDATE")

split_year = 2015  # example: train on pre-2015, test on post-2015

train = df[df["YEAR"] < split_year]
test = df[df["YEAR"] >= split_year]

X_train = train[feature_cols]
y_train = train["LGD"]

X_test = test[feature_cols]
y_test = test["LGD"]


KeyError: 'FEDFUNDS'